# Rentabilidad anual de un alojamiento en Airbnb Bogotá
## Enfoque directo: un modelo para el ingreso

**Pregunta de investigación 3.** Dadas las características de una vivienda, ¿cuál puede ser su
rentabilidad anual como alojamiento de corta estancia?

**Integrantes:** Esteban, Rubén, Andrés · **Universidad EIA**, Maestría · **Profesor:** Daniel Orozco Restrepo

**Datos.** Inside Airbnb, Bogotá, Distrito Capital, Colombia, compilado el 14 de septiembre de
2026. Licencia Creative Commons Attribution 4.0.

---
## La variable de interés

La rentabilidad anual de un alojamiento es el ingreso bruto que produce en un año de
operación, y equivale a la tarifa por noche multiplicada por las noches que consigue vender.

$$\text{ingreso anual} = \text{tarifa por noche} \times \text{noches ocupadas}$$

El enfoque de este cuaderno estima ese ingreso con un solo modelo, a partir de lo que un
propietario conoce antes de publicar. Los dos factores de la identidad quedan excluidos del
conjunto de variables explicativas, junto con todo lo que permita reconstruirlos.

---
# Paso 1 — ¿Machine learning?

## 1.1 Entorno

In [ ]:
import sys                                                   # sys da informacion del interprete en uso
import time                                                  # time mide cuanto tarda cada ajuste
import numpy as np                                           # numpy: operaciones matematicas y arreglos
import pandas as pd                                          # pandas: manejo de tablas
import matplotlib.pyplot as plt                              # matplotlib: graficos
import sklearn                                               # scikit-learn: modelos y preprocesamiento

import airbnb_comun as ac                                    # modulo del proyecto

ac.estilo()                                                  # fija la paleta y el estilo de las figuras
pd.set_option("display.width", 170)                          # ancho de impresion de las tablas
pd.set_option("display.max_columns", 40)                      # columnas maximas que se imprimen
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")  # separador de miles y tres decimales

print("Python       ", sys.version.split()[0])               # version del interprete
print("scikit-learn ", sklearn.__version__)                   # version de scikit-learn
print("Semilla      ", ac.SEMILLA)                            # semilla unica del proyecto

## 1.2 El problema

Un propietario que evalúa alquilar su vivienda por noches necesita el ingreso anual esperado
para compararlo con el arriendo tradicional. Ese ingreso depende de la tarifa que el mercado
acepta y de la fracción del año que consigue vender, y ninguna de las dos se observa antes de
operar.

La dificultad está en que las dos dependen de combinaciones de atributos. Una habitación
privada céntrica sostiene una tarifa baja con ocupación alta, mientras que un apartamento
completo en la misma localidad sostiene una tarifa alta con ocupación menor, y el mismo
apartamento veinte cuadras más al sur cambia las dos cosas a la vez. Un promedio por localidad
asigna el mismo valor a casos cuyo ingreso difiere en un orden de magnitud, y una tabla de
promedios cruzados por localidad, tipo y capacidad deja celdas con muy pocos alojamientos.

## 1.3 Qué es, exactamente, la tarifa publicada

La columna de precio no contiene una tarifa de catálogo. Contiene el resultado de una consulta
de disponibilidad, con su fecha, su duración y su régimen de precio, y la fuente conserva esa
consulta en un objeto que acompaña a cada anuncio. Leerlo determina qué significa la cifra.

In [ ]:
listings = ac.cargar_listings()                              # carga y abre la cotizacion de cada anuncio
print("Filas y columnas:", listings.shape)                   # dimensiones de la tabla

ejemplo = listings.loc[listings["es_mensual"], ["price", "noches_cotizadas", "descuento_cotizacion"]].head(1)
print()
print("Un anuncio cuyo precio por noche procede de una tarifa mensual:")
print(ejemplo.to_string())
print()
import json                                                   # json interpreta el objeto de la cotizacion
crudo = json.loads(listings.loc[ejemplo.index[0], "price_quote_raw"])["quote"]   # objeto de esa cotizacion
for linea in crudo["raw_price_line_items"]:                   # recorre el desglose que devolvio la consulta
    print(f"   {linea['description']:32s} {linea['price_string']}")
print(f"   {'precio por noche resultante':32s} ${float(crudo['price_per_night']):,.2f}")
print(f"   {'estancia consultada':32s} {crudo['requested_checkin_date']} a {crudo['requested_checkout_date']}")

El desglose muestra el mecanismo. Para los anuncios con estancia mínima larga, la consulta
devuelve un precio mensual promedio, le resta los descuentos de estancia larga y reparte el
resultado entre las noches del mes. La cifra que queda en la columna de precio es, para ese
grupo, una mensualidad con descuento dividida entre treinta, y no la tarifa de una noche.

La fuente marca el régimen de forma explícita, lo que permite separar los dos productos en
lugar de promediarlos.

In [ ]:
con_tarifa = listings[listings["precio_noche"].notna()]      # anuncios con tarifa publicada

regimen = con_tarifa.groupby("es_mensual").agg(              # compara los dos regimenes de precio
    anuncios=("id", "size"),
    tarifa_mediana=("precio_noche", "median"),
    huespedes_mediana=("accommodates", "median"),
    noches_cotizadas=("noches_cotizadas", "median"),
    estancia_minima=("minimum_nights", "median"),
    ingreso_mediano=("estimated_revenue_l365d", "median"),
)
regimen.index = ["Tarifa por noche", "Tarifa mensual prorrateada"]   # nombra los dos regimenes
print(regimen.to_string())
print()
print(f"El regimen mensual cubre el {con_tarifa['es_mensual'].mean() * 100:.1f} % de los anuncios con tarifa.")
print(f"Su tarifa mediana es un {(1 - regimen.loc['Tarifa mensual prorrateada', 'tarifa_mediana'] / regimen.loc['Tarifa por noche', 'tarifa_mediana']) * 100:.0f} % menor")
print("con la misma capacidad mediana, y su ingreso anual mediano es cero.")
print()
print(f"Cotizaciones con descuento ya aplicado: {(con_tarifa['descuento_cotizacion'] > 0).mean() * 100:.1f} %")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))          # dos paneles

reparto = con_tarifa["es_mensual"].value_counts().rename({False: "Tarifa por noche", True: "Tarifa mensual"})
barras = ejes[0].barh(reparto.index, reparto.values, color=[ac.ACENTO, ac.MORADO], height=0.55)
for barra, valor in zip(barras, reparto.values):             # etiqueta directa en cada barra
    ejes[0].text(valor + reparto.max() * 0.02, barra.get_y() + barra.get_height() / 2,
                 f"{valor:,} ({valor / len(con_tarifa) * 100:.1f} %)", va="center", fontsize=9, color=ac.TINTA)
ejes[0].set_xlim(0, reparto.max() * 1.3)                      # espacio para las etiquetas
ejes[0].set_title("Regimen de precio de la cotizacion")       # titulo del panel
ejes[0].set_xlabel("anuncios")                                 # etiqueta del eje horizontal

for etiqueta, color, mascara in [("Tarifa por noche", ac.ACENTO, ~con_tarifa["es_mensual"]),
                                 ("Tarifa mensual", ac.MORADO, con_tarifa["es_mensual"])]:
    ejes[1].hist(np.log1p(con_tarifa.loc[mascara, "precio_noche"]), bins=55, alpha=0.62,
                 color=color, label=etiqueta, edgecolor="white", linewidth=0.3)
ejes[1].set_title("Tarifa por noche segun el regimen")        # titulo del panel
ejes[1].set_xlabel("log(1 + tarifa por noche)")                # etiqueta del eje horizontal
ejes[1].set_ylabel("alojamientos")                             # etiqueta del eje vertical
ejes[1].legend(fontsize=9)                                     # leyenda, porque hay dos series

plt.tight_layout()                                             # acomoda los paneles
plt.show()                                                     # muestra la figura

## 1.4 El objetivo y sus tres poblaciones

El ingreso que publica la fuente se estima a partir del número de comentarios recibidos. Esa
construcción produce tres grupos que conviene distinguir antes de modelar, porque la cifra
significa algo distinto en cada uno.

Los anuncios sin comentarios en la ventana registran ingreso cero por definición del método, de
modo que el cero mezcla al que nunca se alquiló con el que se alquiló y no recibió reseña. Los
anuncios publicados dentro de la ventana tienen una exposición menor a un año, y su ingreso
anual no es comparable con el de un anuncio que estuvo publicado los doce meses. Los anuncios
más activos quedan en el techo de la estimación de ocupación.

In [ ]:
datos = ac.preparar_poblacion(listings)                      # agrega exposicion, actividad e ingreso por dia
reglas_cal = ac.reglas_calendario()                          # reglas de reserva que el anfitrion fija dia a dia
datos = datos.join(reglas_cal, on="id")                      # las pega a cada alojamiento por su identificador
print(f"Reglas del calendario unidas para {len(reglas_cal):,} alojamientos")
print(f"Cambian su estancia minima durante el ano: {datos['varia_estancia_minima'].mean() * 100:.1f} %")
datos = datos[datos["precio_noche"].notna()].copy()           # conserva los anuncios con tarifa publicada

print(f"Anuncios con tarifa publicada: {len(datos):,}")
print()
print(f"  sin actividad detectada en la ventana:   {(~datos['actividad_detectada']).mean() * 100:5.1f} %")
print(f"  sin comentario alguno, nunca:            {(~datos['exposicion_observable']).mean() * 100:5.1f} %")
print(f"  con exposicion menor a un ano:           {(datos['exposicion_observable'] & ~datos['exposicion_completa']).mean() * 100:5.1f} %")
print(f"  en el techo de la estimacion de noches:  {(datos['noches_ocupadas'] >= 255).mean() * 100:5.1f} %")
print()
sin_resena = datos[~datos["exposicion_observable"]]           # anuncios que nunca recibieron comentario
print(f"De los {len(sin_resena):,} anuncios sin comentario alguno,")
print(f"los que registran ingreso mayor que cero son: {(sin_resena['ingreso_anual'] > 0).sum()}")
print()
print("El cero no es una medida del mercado: es el valor que toma la estimacion")
print("cuando el metodo no detecta actividad.")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13.5, 3.6))          # tres paneles

tope_vista = datos["ingreso_anual"].quantile(0.99)           # recorta la cola para que el histograma sea legible
ejes[0].hist(datos.loc[datos["ingreso_anual"] <= tope_vista, "ingreso_anual"], bins=60,
             color=ac.ACENTO, edgecolor="white", linewidth=0.3)
ejes[0].set_title("Ingreso anual en pesos")                   # titulo del panel
ejes[0].set_xlabel("pesos por ano, hasta el percentil 99")     # etiqueta del eje horizontal
ejes[0].set_ylabel("alojamientos")                             # etiqueta del eje vertical
ejes[0].xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))  # separador de miles

ejes[1].hist(np.log1p(datos["ingreso_anual"].dropna()), bins=60, color=ac.MORADO,
             edgecolor="white", linewidth=0.3)
ejes[1].set_title("Ingreso anual en escala logaritmica")      # titulo del panel
ejes[1].set_xlabel("log(1 + ingreso anual)")                   # etiqueta del eje horizontal

composicion = pd.Series({                                     # composicion segun el estado del anuncio
    "Sin comentario alguno": (~datos["exposicion_observable"]).sum(),
    "Con historial, sin\nactividad en la ventana": (datos["exposicion_observable"] & ~datos["actividad_detectada"]).sum(),
    "Exposicion parcial": (datos["actividad_detectada"] & ~datos["exposicion_completa"]).sum(),
    "Exposicion completa\ny activo": (datos["actividad_detectada"] & datos["exposicion_completa"]).sum(),
})
colores = [ac.GRIS, ac.GRIS, ac.MORADO, ac.ACENTO]            # gris para lo no medible
barras = ejes[2].barh(composicion.index, composicion.values, color=colores, height=0.62)
for barra, valor in zip(barras, composicion.values):          # etiqueta directa en cada barra
    ejes[2].text(valor + composicion.max() * 0.02, barra.get_y() + barra.get_height() / 2,
                 f"{valor / len(datos) * 100:.0f} %", va="center", fontsize=8.5, color=ac.TINTA)
ejes[2].set_xlim(0, composicion.max() * 1.25)                 # espacio para las etiquetas
ejes[2].invert_yaxis()                                         # la primera categoria arriba
ejes[2].tick_params(axis="y", labelsize=8)                     # etiquetas mas pequenas
ejes[2].set_title("Estado de medicion del objetivo")           # titulo del panel
ejes[2].set_xlabel("alojamientos")                              # etiqueta del eje horizontal

plt.tight_layout()                                              # acomoda los paneles
plt.show()                                                      # muestra la figura

## 1.5 Partición de entrenamiento y prueba

Un anfitrión puede publicar decenas de anuncios casi idénticos en el mismo edificio, con la
misma capacidad y la misma tarifa. Una partición aleatoria por filas reparte esos anuncios a
ambos lados, de modo que el modelo recupera en prueba una tarifa que ya vio en entrenamiento y
el error medido deja de reflejar la capacidad de predecir un alojamiento nuevo.

La partición agrupa por anfitrión. Todos los anuncios de un mismo anfitrión caen del mismo
lado, y el error en prueba corresponde a operadores que el modelo no vio.

In [ ]:
tamanos = datos["host_id"].value_counts()                     # anuncios por anfitrion
print(f"Anuncios: {len(datos):,}   Anfitriones distintos: {datos['host_id'].nunique():,}")
print(f"Anuncios por anfitrion: mediana {tamanos.median():.0f}, maximo {tamanos.max()}")
print(f"Anuncios de anfitriones con cinco o mas: {datos['host_id'].map(tamanos).ge(5).mean() * 100:.1f} %")

llave = ["host_id", "room_type", "accommodates", "neighbourhood_cleansed", "precio_noche"]   # define un gemelo
gemelos = datos.duplicated(subset=llave, keep=False)          # anuncios con otro identico en esas cinco columnas
print(f"Anuncios con un gemelo del mismo anfitrion: {gemelos.mean() * 100:.1f} %")
print()
mayor = tamanos.index[0]                                       # el anfitrion con mas anuncios
print(f"Ejemplo, el anfitrion con mas anuncios ({tamanos.iloc[0]} en total):")
print(datos[datos["host_id"] == mayor][["name", "accommodates", "neighbourhood_cleansed", "precio_noche"]].head(4).to_string(index=False))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit         # particion que respeta los grupos

particion = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=ac.SEMILLA)   # una particion 80 / 20
indices_train, indices_test = next(particion.split(datos, groups=datos["host_id"]))   # agrupando por anfitrion
train = datos.iloc[indices_train].copy()                      # particion de entrenamiento
test = datos.iloc[indices_test].copy()                        # particion de prueba

print(f"Entrenamiento: {len(train):,} anuncios de {train['host_id'].nunique():,} anfitriones")
print(f"Prueba:        {len(test):,} anuncios de {test['host_id'].nunique():,} anfitriones")
print(f"Anfitriones en ambas particiones: {len(set(train['host_id']) & set(test['host_id']))}")

## 1.6 Regla a mano

Sin modelo, el propietario dispone del ingreso típico de su localidad.

In [ ]:
from sklearn.dummy import DummyRegressor                      # modelo que siempre predice el mismo valor

con_objetivo = train[train["ingreso_anual"].notna()]          # entrenamiento con el objetivo medido
prueba_objetivo = test[test["ingreso_anual"].notna()]         # prueba con el objetivo medido

regla = ac.ReglaPorGrupo("neighbourhood_cleansed").fit(con_objetivo, con_objetivo["ingreso_anual"])
base = DummyRegressor(strategy="median").fit(con_objetivo, con_objetivo["ingreso_anual"])

referencia = pd.DataFrame({                                    # tabla de los dos puntos de partida
    "Linea base (mediana de la ciudad)": ac.metricas(prueba_objetivo["ingreso_anual"], base.predict(prueba_objetivo)),
    "Regla a mano (mediana por localidad)": ac.metricas(prueba_objetivo["ingreso_anual"], regla.predict(prueba_objetivo)),
}).T
print(referencia.to_string())
print()
print(f"La regla a mano se equivoca en ${referencia.loc['Regla a mano (mediana por localidad)', 'MAE']:,.0f} al ano por alojamiento,")
print(f"sobre un ingreso mediano observado de ${prueba_objetivo['ingreso_anual'].median():,.0f}.")

---
# Paso 2 — Datos

## 2.1 Fuente

Los datos provienen de Inside Airbnb, un proyecto que publica extracciones periódicas de los
anuncios visibles en el sitio de Airbnb. La extracción corresponde a Bogotá, Distrito Capital,
Colombia, compilada el 14 de septiembre de 2026, y se distribuye bajo licencia Creative Commons
Attribution 4.0, que permite el uso con atribución de la fuente.

La descarga entrega tres archivos comprimidos que se cruzan por el identificador del
alojamiento. El papel de cada uno difiere: la tabla de anuncios aporta las variables
explicativas, mientras que el calendario y los comentarios construyen y verifican el objetivo.
La ocupación observada describe el resultado de operar, de modo que no puede entrar como
variable explicativa de un modelo dirigido a quien todavía no opera.

In [ ]:
calendario_cols = pd.read_csv(ac.DATOS / "calendar.csv.gz", nrows=0).columns   # solo el encabezado
resenas_cols = pd.read_csv(ac.DATOS / "reviews.csv.gz", nrows=0).columns       # solo el encabezado

print("listings :", f"{listings.shape[0]:,}", "filas  |  una fila por alojamiento")
print("calendar : 7,119,325 filas  |  una fila por alojamiento y dia")
print("reviews  :   568,675 filas  |  una fila por comentario")
print()
print("columnas de calendar:", list(calendario_cols))         # el calendario no trae columna de precio
print("columnas de reviews :", list(resenas_cols))

In [ ]:
fig, ax = ac.lienzo(12, 4.3)                                  # lienzo para el esquema

ac.caja(ax, 0.02, 0.60, 0.20, 0.30, "listings", "19.505 filas\nuna por alojamiento", ac.ACENTO)
ac.caja(ax, 0.02, 0.22, 0.20, 0.30, "calendar", "7.119.325 filas\nalojamiento x dia", ac.MORADO)
ac.caja(ax, 0.02, -0.16, 0.20, 0.30, "reviews", "568.675 filas\nun comentario", ac.MORADO)

ac.caja(ax, 0.34, 0.60, 0.22, 0.30, "Variables explicativas", "lo que se conoce\nantes de operar", ac.ACENTO)
ac.caja(ax, 0.34, 0.03, 0.22, 0.42, "Construccion y verificacion\ndel objetivo",
        "exposicion, actividad\ny ocupacion", ac.MORADO)
ac.caja(ax, 0.68, 0.33, 0.24, 0.40, "Tabla de analisis", "una fila por alojamiento\nobjetivo: ingreso", ac.TINTA)

ac.flecha(ax, (0.22, 0.75), (0.34, 0.75))                     # listings a variables explicativas
ac.flecha(ax, (0.22, 0.37), (0.34, 0.30))                     # calendar a verificacion
ac.flecha(ax, (0.22, -0.01), (0.34, 0.16))                    # reviews a verificacion
ac.flecha(ax, (0.56, 0.75), (0.68, 0.60))                     # a la tabla de analisis
ac.flecha(ax, (0.56, 0.24), (0.68, 0.45))                     # a la tabla de analisis

ax.set_ylim(-0.2, 0.95)                                        # ajusta el alto
ax.set_title("Papel de cada tabla en el analisis", loc="left", pad=12)
plt.show()                                                     # muestra el esquema

## 2.2 Verificación del objetivo

Dos comprobaciones fijan el significado del ingreso publicado. La primera divide el ingreso
entre las noches ocupadas y lo compara con la tarifa. La segunda reconstruye la ocupación desde
el número de comentarios, que es el procedimiento con que la fuente la estima.

In [ ]:
activos = train[train["noches_ocupadas"] > 0]                 # anuncios con al menos una noche ocupada
razon = activos["ingreso_anual"] / activos["noches_ocupadas"]  # ingreso dividido entre noches
coincide = np.isclose(razon, activos["precio_noche"], rtol=0.001).mean() * 100   # coincidencia con la tarifa

print(f"El ingreso dividido entre las noches ocupadas reproduce la tarifa publicada")
print(f"en el {coincide:.1f} % de los anuncios con actividad.")
print()
print(pd.DataFrame({"ingreso_anual": activos["ingreso_anual"].head(4),
                    "noches_ocupadas": activos["noches_ocupadas"].head(4),
                    "razon": razon.head(4),
                    "tarifa": activos["precio_noche"].head(4)}).to_string())

In [ ]:
t0 = time.time()                                               # marca el inicio de la carga
fraccion_bloqueada, fecha_corte = ac.ocupacion_calendario()    # resume el calendario del ano siguiente
noches_resenas = ac.ocupacion_reviews(fecha_corte)             # reconstruye la ocupacion desde los comentarios
print(f"Fecha de corte de los datos: {fecha_corte.date()}   ({time.time() - t0:.0f} s de carga)")

verificacion = pd.DataFrame({                                  # las tres medidas de ocupacion
    "noches_publicadas": train["noches_ocupadas"],
    "noches_por_resenas": train["id"].map(noches_resenas).fillna(0),
    "fraccion_bloqueada": train["id"].map(fraccion_bloqueada),
})
print()
print("Correlacion de rangos entre las tres medidas:")
print(verificacion.corr(method="spearman").to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))           # dos paneles

ejes[0].scatter(verificacion["noches_por_resenas"], verificacion["noches_publicadas"],
                s=5, alpha=0.18, color=ac.MORADO, edgecolors="none")
limite = 300                                                   # tope de los ejes, por encima del techo de 255
ejes[0].plot([0, limite], [0, limite], color=ac.TINTA, linewidth=1, linestyle="--")
ejes[0].set_xlim(0, limite); ejes[0].set_ylim(0, limite)       # mismos limites en los dos ejes
ejes[0].set_title("Ocupacion reconstruida y publicada")         # titulo del panel
ejes[0].set_xlabel("noches estimadas desde los comentarios")     # etiqueta del eje horizontal
ejes[0].set_ylabel("noches publicadas por la fuente")            # etiqueta del eje vertical
ejes[0].text(limite * 0.05, limite * 0.9, "linea de identidad", fontsize=8, color=ac.TINTA)

mensual = (fraccion_bloqueada.rename("bloqueada").to_frame()
           .join(pd.Series(1, index=fraccion_bloqueada.index, name="n")))   # marco para el segundo panel
calendario_mes = ac.DATOS                                       # referencia a la carpeta, sin recargar el archivo
ejes[1].scatter(verificacion["fraccion_bloqueada"], verificacion["noches_publicadas"],
                s=5, alpha=0.18, color=ac.ACENTO, edgecolors="none")
ejes[1].set_title("Calendario futuro y ocupacion publicada")     # titulo del panel
ejes[1].set_xlabel("fraccion de los 365 dias siguientes no disponible")
ejes[1].set_ylabel("noches publicadas por la fuente")

plt.tight_layout()                                               # acomoda los paneles
plt.show()                                                       # muestra la figura

La reconstrucción desde los comentarios reproduce la ocupación publicada, lo que confirma por
qué vía se construye el objetivo y, con ello, qué columnas quedan prohibidas.

El calendario, en cambio, no la verifica, y conviene detenerse en el motivo porque invita a un
error de lectura. La ocupación estimada mira los trescientos sesenta y cinco días anteriores a
la extracción, mientras que el calendario describe los trescientos sesenta y cinco siguientes,
que todavía nadie reservó. Las reservas se acumulan a medida que se acerca la fecha, de modo
que la fracción bloqueada decae con la distancia y un alojamiento que vendió el año entero
aparece con el calendario casi vacío para el año próximo.

A esa confusión temporal se suma una de significado. Una noche marcada como no disponible
puede estar vendida o puede estar bloqueada porque el anfitrión vive allí y se la reserva para
sí, y las dos situaciones son indistinguibles en el archivo. El calendario queda entonces como
descripción del comportamiento de la oferta, no como medida del objetivo.

In [ ]:
calendario = pd.read_csv(ac.DATOS / "calendar.csv.gz", usecols=["listing_id", "date", "available"])
calendario["date"] = pd.to_datetime(calendario["date"])       # convierte la fecha de texto a fecha real
mes = calendario["date"].dt.to_period("M").astype(str)        # mes de cada dia del calendario
bloqueo = (calendario["available"] == "f").groupby(mes).mean() * 100   # porcentaje bloqueado por mes

abierta = (calendario["available"] == "t").groupby(calendario["listing_id"]).mean()   # fraccion ofrecida del ano
train["fraccion_ofrecida"] = train["id"].map(abierta)         # la pega a cada anuncio de entrenamiento

fig, ejes = plt.subplots(1, 2, figsize=(12, 3.9))             # dos paneles

ejes[0].plot(range(len(bloqueo)), bloqueo.values, "o-", color=ac.ACENTO, linewidth=2, markersize=6)
ejes[0].set_xticks(range(len(bloqueo)))                        # una marca por mes
ejes[0].set_xticklabels([m[2:] for m in bloqueo.index], fontsize=8, rotation=45)
ejes[0].set_title("Noches no disponibles, por mes del calendario")
ejes[0].set_xlabel("mes"); ejes[0].set_ylabel("porcentaje de noches")
ejes[0].text(1.2, bloqueo.max() * 0.82, "las reservas se acumulan\nal acercarse la fecha",
             fontsize=8.5, color=ac.TINTA)

tramos = pd.cut(train["noches_ocupadas"], [0, 30, 90, 180, 255], include_lowest=True)
oferta = train.groupby(tramos, observed=True)["fraccion_ofrecida"].median() * 100
barras = ejes[1].bar(range(len(oferta)), oferta.values, color=ac.MORADO, width=0.6)
ejes[1].set_xticks(range(len(oferta)))
ejes[1].set_xticklabels([str(i) for i in oferta.index], fontsize=8)
for barra, valor in zip(barras, oferta.values):                # etiqueta directa en cada barra
    ejes[1].text(barra.get_x() + barra.get_width() / 2, valor + 1, f"{valor:.0f} %",
                 ha="center", fontsize=9, color=ac.TINTA)
ejes[1].set_ylim(0, 110)
ejes[1].set_title("Fraccion del ano siguiente que se ofrece")
ejes[1].set_xlabel("noches ocupadas el ano anterior"); ejes[1].set_ylabel("porcentaje ofrecido")

plt.tight_layout()
plt.show()

In [ ]:
sin_ingreso = train[train["ingreso_anual"].fillna(0) == 0]     # anuncios sin ingreso detectado
abiertos = (sin_ingreso["fraccion_ofrecida"] > 0.5).mean() * 100   # cuantos ofrecen mas de medio ano

print(f"De los {len(sin_ingreso):,} anuncios sin ingreso detectado,")
print(f"el {abiertos:.1f} % ofrece mas de medio ano de calendario.")
print()
print("El cero no corresponde a inventario retirado del mercado: corresponde")
print("a alojamientos que se ofrecen y no consiguen venderse, o cuya venta")
print("el metodo no detecta por falta de comentarios.")
print()
comparacion = train.assign(vende=train["ingreso_anual"].fillna(0) > 0).groupby("vende").agg(
    anuncios=("id", "size"),
    superanfitrion_pct=("host_is_superhost", lambda s: round((s == "t").mean() * 100, 1)),
    habitacion_privada_pct=("room_type", lambda s: round((s == "Private room").mean() * 100, 1)),
    ofrecido_mediano=("fraccion_ofrecida", "median"),
)
comparacion.index = ["No registra ingreso", "Registra ingreso"]
print(comparacion.to_string())

## 2.3 Exploración de las columnas

La tabla de anuncios trae noventa columnas, y una parte llega vacía o constante desde el
origen.

In [ ]:
originales = [c for c in listings.columns if c in train.columns]   # columnas de la tabla original
resumen = pd.DataFrame({                                       # una fila por columna
    "tipo": train[originales].dtypes.astype(str),
    "vacios_%": (train[originales].isna().mean() * 100).round(1),
    "unicos": train[originales].nunique(dropna=True),
})
print(f"Columnas revisadas:            {len(resumen)}")
print(f"  vacias por completo:         {(resumen['vacios_%'] == 100).sum()}")
print(f"  con un solo valor distinto:  {(resumen['unicos'] <= 1).sum()}")
print()
print("Vacias por completo:")
print("  " + ", ".join(resumen.index[resumen["vacios_%"] == 100]))
ruta = ac.guardar(resumen, "auditoria_columnas.csv")           # guarda la auditoria completa
print()
print("Auditoria guardada en", ruta.name)

---
# Paso 3 — Limpieza

## 3.1 Formato de la tarifa y duplicados

In [ ]:
print("Formato original:", listings["price"].dropna().head(3).tolist())   # como viene el dato
print("Tras la conversion:", listings["precio_noche"].dropna().head(3).round(0).tolist())
print()
print(f"Anuncios sin tarifa publicada:            {listings['price'].isna().sum():,}")
print(f"Filas identicas en todas las columnas:    {datos.duplicated().sum()}")
print(f"Identificadores de alojamiento repetidos: {datos['id'].duplicated().sum()}")

## 3.2 Separación del régimen mensual

El régimen mensual es un producto distinto medido con un instrumento que no le corresponde. Su
tarifa viene prorrateada y con descuentos de estancia larga ya aplicados, y su ingreso es
invisible para un método que infiere la ocupación de los comentarios, porque el huésped de
estancia larga rara vez reseña. Queda como segmento descriptivo, sin modelo de ingreso, y el
análisis continúa sobre el régimen por noche.

In [ ]:
train = train[~train["es_mensual"]].copy()                    # conserva el regimen por noche en entrenamiento
test = test[~test["es_mensual"]].copy()                       # y en prueba

print(f"Entrenamiento tras separar el regimen mensual: {len(train):,}")
print(f"Prueba tras separar el regimen mensual:        {len(test):,}")

## 3.3 Valores extremos

La tarifa contiene errores de digitación que producen ingresos imposibles, porque el error se
multiplica por el número de noches. El criterio de Tukey (1977) sitúa el corte en el tercer
cuartil más tres veces el rango intercuartílico, aplicado sobre el logaritmo de la tarifa por
huésped. La tarifa por huésped, y no la total, es la base correcta: una vivienda para dieciséis
personas sostiene una tarifa total que en un apartamento para dos sería un error, y el corte
sobre el total elimina alojamientos legítimos de gran capacidad.

In [ ]:
def por_huesped(tabla):                                        # tarifa por huesped de una tabla
    return tabla["precio_noche"] / tabla["accommodates"].clip(lower=1)   # minimo un huesped

log_tarifa = np.log1p(por_huesped(train))                      # logaritmo de la tarifa por huesped
q1, q3 = log_tarifa.quantile([0.25, 0.75])                     # primer y tercer cuartil
tope = float(np.expm1(q3 + 3 * (q3 - q1)))                     # tope de Tukey, devuelto a pesos

print(f"Tope de Tukey sobre la tarifa por huesped: ${tope:,.0f} por noche")
print(f"Por encima del tope: {(por_huesped(train) > tope).sum()} en entrenamiento y {(por_huesped(test) > tope).sum()} en prueba")
print()
print("Los cinco de mayor tarifa, con el ingreso que implicarian:")
print(train.nlargest(5, "precio_noche")[["precio_noche", "accommodates", "noches_ocupadas", "ingreso_anual"]].to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))           # dos paneles

ejes[0].hist(log_tarifa, bins=70, color=ac.ACENTO, edgecolor="white", linewidth=0.3)
ejes[0].axvline(np.log1p(tope), color=ac.TINTA, linewidth=1.3, linestyle="--")
ejes[0].text(np.log1p(tope), ejes[0].get_ylim()[1] * 0.9, f"  tope de Tukey\n  ${tope:,.0f}",
             fontsize=8, color=ac.TINTA, va="top")
ejes[0].set_title("Tarifa por huesped en escala logaritmica")
ejes[0].set_xlabel("log(1 + tarifa por huesped)")
ejes[0].set_ylabel("alojamientos")

conservados = train[por_huesped(train) <= tope]                # los que se conservan
descartados = train[por_huesped(train) > tope]                 # los que se descartan
ejes[1].scatter(conservados["accommodates"], conservados["precio_noche"], s=6, alpha=0.2,
                color=ac.MORADO, edgecolors="none", label="conservados")
ejes[1].scatter(descartados["accommodates"], descartados["precio_noche"], s=26, color=ac.ACENTO,
                edgecolors=ac.TINTA, linewidth=0.4, label="descartados")
ejes[1].set_yscale("log")                                      # escala logaritmica en el eje vertical
ejes[1].set_title("Tarifa y capacidad")
ejes[1].set_xlabel("huespedes que admite")
ejes[1].set_ylabel("tarifa por noche, escala log")
ejes[1].legend(loc="lower right", fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
train = train[por_huesped(train) <= tope].copy()               # aplica el corte a entrenamiento
test = test[por_huesped(test) <= tope].copy()                  # y a prueba, porque el objetivo esta mal medido alli

print(f"Entrenamiento: {len(train):,} | Prueba: {len(test):,}")

## 3.4 Valores faltantes

La estrategia de relleno se decide comparando el error de cada opción. El procedimiento esconde
una parte de los valores conocidos, los reconstruye con cada método y mide el error absoluto
medio contra el valor verdadero.

In [ ]:
from sklearn.ensemble import RandomForestRegressor            # una de las opciones de relleno
from sklearn.metrics import mean_absolute_error               # error absoluto medio

def comparar_relleno(columna, grupo):                         # compara cuatro formas de rellenar
    conocidos = train[train[columna].notna()]                 # filas donde el valor existe
    muestra = conocidos.sample(frac=0.2, random_state=ac.SEMILLA)   # esconde el 20 %
    resto = conocidos.drop(muestra.index)                     # el 80 % restante sirve para aprender
    verdad = muestra[columna]                                  # valores verdaderos escondidos

    opciones = {
        "Media": pd.Series(resto[columna].mean(), index=muestra.index),
        "Mediana global": pd.Series(resto[columna].median(), index=muestra.index),
        "Mediana por grupo": muestra[grupo].map(resto.groupby(grupo)[columna].median()).fillna(resto[columna].median()),
    }
    apoyo = ["accommodates", "latitude", "longitude"]           # variables completas con que predecir
    bosque = RandomForestRegressor(n_estimators=120, random_state=ac.SEMILLA).fit(resto[apoyo], resto[columna])
    opciones["Regresion (bosque)"] = pd.Series(bosque.predict(muestra[apoyo]), index=muestra.index)
    return {nombre: mean_absolute_error(verdad, valores) for nombre, valores in opciones.items()}

relleno = pd.DataFrame({
    "beds (grupo: accommodates)": comparar_relleno("beds", "accommodates"),
    "bedrooms (grupo: accommodates)": comparar_relleno("bedrooms", "accommodates"),
}).T
print(relleno.to_string())
print()
for fila in relleno.index:                                     # declara el ganador de cada columna
    mejor = relleno.loc[fila].idxmin()
    print(f"{fila}: menor error con {mejor.lower()}, MAE {relleno.loc[fila, mejor]:.3f} "
          f"contra {relleno.loc[fila, 'Media']:.3f} de la media")

La mediana por grupo de capacidad resulta la opción de menor error en las dos columnas. El
mecanismo es directo: el número de camas y de habitaciones queda determinado casi por completo
por el número de huéspedes que el alojamiento admite, de modo que condicionar en esa variable
recupera casi toda la información perdida, mientras que la media de la ciudad ignora el tamaño.

## 3.5 Exposición

El ingreso publicado agrega trescientos sesenta y cinco días, pero no todos los anuncios
estuvieron publicados ese tiempo. El objetivo se expresa como tasa por día expuesto, lo que
restituye la comparabilidad sin eliminar observaciones. La exposición solo es observable para
los anuncios con al menos un comentario, porque el archivo no trae la fecha de publicación.

In [ ]:
observable = train[train["exposicion_observable"]]            # anuncios con exposicion medible
print(f"Exposicion observable: {len(observable):,} de {len(train):,} anuncios de entrenamiento")
print(f"Exposicion mediana:    {observable['dias_expuesto'].median():.0f} dias")
print(f"Con menos de un ano:   {(~observable['exposicion_completa']).mean() * 100:.1f} %")
print()
print("Ingreso por dia expuesto, entre los anuncios con actividad detectada:")
activos_obs = observable[observable["actividad_detectada"]]
print(activos_obs["ingreso_por_dia"].describe(percentiles=[.25, .5, .75, .95]).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))           # dos paneles

ejes[0].hist(observable["dias_expuesto"], bins=60, color=ac.MORADO, edgecolor="white", linewidth=0.3)
ejes[0].set_title("Dias de exposicion dentro de la ventana")
ejes[0].set_xlabel("dias publicado en los ultimos 365")
ejes[0].set_ylabel("alojamientos")

ejes[1].scatter(observable["dias_expuesto"], observable["ingreso_anual"], s=6, alpha=0.18,
                color=ac.ACENTO, edgecolors="none")
ejes[1].set_yscale("symlog")                                   # escala logaritmica que admite el cero
ejes[1].set_title("Ingreso anual y exposicion")
ejes[1].set_xlabel("dias de exposicion")
ejes[1].set_ylabel("ingreso anual, escala log")

plt.tight_layout()
plt.show()

## 3.6 Fugas

Una fuga es una columna que solo se conoce después del resultado, o que se calcula con él. El
objetivo obliga a una lista amplia por la forma en que se construye. La tarifa y las noches
ocupadas son los dos factores del producto. La estimación de ocupación se deriva del número de
comentarios, de modo que cualquier columna de comentarios reconstruye el objetivo por una vía
indirecta. La disponibilidad describe el calendario ya operado, y las calificaciones solo
existen cuando el alojamiento ya recibió huéspedes.

La fecha de la cotización merece mención aparte. La consulta pidió la primera fecha con
disponibilidad, de modo que la distancia hasta esa fecha mide cuánto lleva cerrado el
calendario, que es una característica del resultado y no del inmueble.

In [ ]:
fig, ax = ac.lienzo(12, 3.9)                                  # lienzo para el esquema de fugas

ac.caja(ax, 0.38, 0.72, 0.24, 0.24, "Ingreso anual", "variable objetivo", ac.TINTA)
ac.caja(ax, 0.12, 0.40, 0.22, 0.22, "Tarifa por noche", "", ac.ACENTO)
ac.caja(ax, 0.66, 0.40, 0.22, 0.22, "Noches ocupadas", "", ac.ACENTO)
ac.caja(ax, 0.60, 0.04, 0.34, 0.24, "Numero de comentarios", "insumo de la ocupacion estimada", ac.MORADO)
ac.caja(ax, 0.04, 0.04, 0.34, 0.24, "Cotizacion y disponibilidad", "reproducen tarifa y calendario", ac.MORADO)

ac.flecha(ax, (0.23, 0.62), (0.44, 0.72), "x", ac.TINTA)
ac.flecha(ax, (0.77, 0.62), (0.56, 0.72), "", ac.TINTA)
ac.flecha(ax, (0.77, 0.28), (0.77, 0.40))
ac.flecha(ax, (0.21, 0.28), (0.21, 0.40))
ax.text(0.5, 0.60, "identidad contable", ha="center", fontsize=8.5, color=ac.TINTA, style="italic")
ax.set_title("Origen de la lista de columnas prohibidas", loc="left", pad=10)
plt.show()

In [ ]:
descartes = ac.tabla_descartes()                               # tabla de columnas descartadas
print("Columnas descartadas, por grupo:")
print(descartes["Grupo"].value_counts().to_string())
print()
print("Descartadas por fuga:")
print(descartes[descartes["Grupo"] == "Fuga"][["Columna", "Motivo"]].head(12).to_string(index=False))
ruta = ac.guardar(descartes, "columnas_descartadas.csv")
print()
print(f"Tabla completa, {len(descartes)} columnas, guardada en {ruta.name}")

---
# Paso 4 — Columnas

## 4.1 Selección por mecanismo

La selección declara primero el mecanismo por el que se espera que cada variable afecte al
ingreso, y el signo de ese efecto, antes de medir cualquier asociación. Una selección guiada
por la asociación observada sobre los mismos datos con que después se entrena infla el
desempeño aparente y descarta variables cuyo efecto no es monótono.

Las variables quedan agrupadas por tema, y cada tema corresponde a una decisión distinta del
propietario. La estructura y el equipamiento describen el inmueble; la ubicación, el mercado al
que accede; las reglas de reserva, el modelo de negocio que elige; el anfitrión, la capacidad
de gestión; la presentación, la calidad del anuncio; y la tipología, el producto que vende.

In [ ]:
variables = ac.tabla_variables()                               # tabla de variables con su mecanismo
print(f"Variables declaradas: {len(variables)} en {variables['Tema'].nunique()} temas")
print()
print(pd.crosstab(variables["Tema"], variables["Quien la controla"]).to_string())

## 4.2 Quién controla cada variable

La pregunta la formula un propietario que todavía no publicó, y eso parte el conjunto de
variables en tres grupos según quién las determina.

La dotación es lo que ya posee y no elige, como la ubicación, el número de habitaciones o los
metros. Las decisiones son aquello a lo que puede comprometerse antes de publicar, desde
arrendar una habitación o la vivienda entera hasta la estancia mínima que exigirá, las
comodidades que instalará o si se registrará ante la autoridad de turismo. El desempeño es
consecuencia de haber operado, y no existe para quien todavía no empezó.

El tercer grupo queda fuera del modelo. El motivo no es una fuga formal, sino que esas
variables toman siempre el mismo valor en todas las predicciones que este propietario va a
pedir, de modo que no informan su decisión y solo mejoran el ajuste sobre la población que ya
opera. El distintivo de superanfitrión añade una razón propia: exige diez estancias
completadas en la misma ventana de doce meses con que se mide la actividad, de modo que
tenerlo implica la etiqueta que la primera etapa quiere predecir.

In [ ]:
for grupo in ["Dotacion", "Decision", "Desempeno"]:            # recorre los tres grupos de control
    bloque = variables[variables["Quien la controla"] == grupo]
    print("=" * 96)
    print(f"  {grupo.upper()}   ({len(bloque)} variables)")
    print("=" * 96)
    for _, fila in bloque.iterrows():
        print(f"  {fila['Variable']:28s} {fila['Tema']:20s} signo esperado: {fila['Signo esperado']}")
    print()

print("Bloques que entran en el modelo del propietario nuevo:")
print("  " + ", ".join(ac.BLOQUES_ELEGIDOS))
print("Bloques que se anaden en el escenario de un operador con historial:")
print("  " + ", ".join(ac.DESEMPENO))

In [ ]:
for tema in variables["Tema"].unique():                        # recorre los temas en el orden declarado
    bloque = variables[variables["Tema"] == tema]
    print("=" * 104)
    print(f"  {tema.upper()}")
    print("=" * 104)
    for _, fila in bloque.iterrows():                          # imprime cada variable con su mecanismo
        print(f"\n  {fila['Variable']}   [{fila['Origen']}]   signo esperado: {fila['Signo esperado']}")
        texto = fila["Relacion teorica con el ingreso anual"]
        while texto:                                            # parte el texto en lineas legibles
            corte = texto[:96].rfind(" ") if len(texto) > 96 else len(texto)
            print("      " + texto[:corte].strip())
            texto = texto[corte:].strip()
    print()
ac.guardar(variables, "variables_seleccionadas.csv")

## 4.3 Señal medida

La asociación se mide de dos formas porque cada una detecta algo distinto. La correlación de
rangos de Spearman recoge la asociación monótona, mientras que la información mutua recoge
cualquier dependencia, incluida la que cambia de signo a lo largo del recorrido de la variable.

La información mutua estimada por vecinos más cercanos crece con el número de valores distintos
de la variable, de modo que sirve como tamiz de relaciones no monótonas y no como medida
comparable entre variables de cardinalidad muy distinta. La confirmación llega en el paso seis,
con la importancia por permutación sobre el modelo ya ajustado.

In [ ]:
from sklearn.feature_selection import mutual_info_regression   # informacion mutua para objetivo continuo

preparado = ac.CrearColumnas().fit_transform(train)            # crea las columnas derivadas
preparado = ac.DummiesAmenities().fit(preparado).transform(preparado)   # abre las comodidades

numericas = []                                                  # variables numericas a medir
for nombre in ac.BLOQUES_ELEGIDOS:                             # recorre los bloques elegidos
    numericas += ac.BLOQUES.get(nombre, [])                    # acumula sus columnas
numericas = [c for c in dict.fromkeys(numericas) if c in preparado.columns]   # sin repetidas

medible = preparado[preparado["ingreso_anual"].notna()]         # filas con el objetivo medido
objetivo = np.log1p(medible["ingreso_anual"])                   # objetivo en escala logaritmica
matriz = medible[numericas].apply(lambda s: s.fillna(s.median()))   # relleno solo para poder medir

senal = pd.DataFrame({
    "rho_Spearman": [medible[c].corr(objetivo, method="spearman") for c in numericas],
    "info_mutua": mutual_info_regression(matriz, objetivo, random_state=ac.SEMILLA),
}, index=numericas).sort_values("info_mutua", ascending=False)

print(f"Variables numericas medidas: {len(senal)}")
print()
print(senal.head(18).round(4).to_string())

In [ ]:
orden = senal.copy()                                            # copia para comparar los dos criterios
orden["puesto_rho"] = orden["rho_Spearman"].abs().rank(ascending=False)    # puesto segun correlacion
orden["puesto_mi"] = orden["info_mutua"].rank(ascending=False)             # puesto segun informacion mutua
orden["salto"] = orden["puesto_rho"] - orden["puesto_mi"]                  # cuanto sube al cambiar de criterio
caso = orden.sort_values("salto", ascending=False).index[0]                # la variable que mas sube

print("Variables que un filtro por correlacion habria descartado:")
print(orden.sort_values("salto", ascending=False).head(5)[
    ["rho_Spearman", "info_mutua", "puesto_rho", "puesto_mi"]].round(4).to_string())
print()
print(f"El caso mas marcado es {caso}, con correlacion de rangos {senal.loc[caso, 'rho_Spearman']:.3f},")
print(f"que la deja en el puesto {int(orden.loc[caso, 'puesto_rho'])} de {len(senal)}, frente a una")
print(f"informacion mutua de {senal.loc[caso, 'info_mutua']:.3f}, que la deja en el puesto {int(orden.loc[caso, 'puesto_mi'])}.")

In [ ]:
top = senal.head(16).iloc[::-1]                                 # las 16 de mayor informacion mutua
fig, ejes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)    # dos paneles con el mismo eje vertical

ejes[0].hlines(top.index, 0, top["info_mutua"], color=ac.GRIS, linewidth=1)
ejes[0].plot(top["info_mutua"], top.index, "o", color=ac.ACENTO, markersize=7)
ejes[0].set_title("Informacion mutua con el ingreso")
ejes[0].set_xlabel("informacion mutua")

ejes[1].hlines(top.index, 0, top["rho_Spearman"], color=ac.GRIS, linewidth=1)
ejes[1].plot(top["rho_Spearman"], top.index, "o", color=ac.MORADO, markersize=7)
ejes[1].axvline(0, color=ac.TINTA, linewidth=0.8)
ejes[1].set_title("Correlacion de rangos con el ingreso")
ejes[1].set_xlabel("rho de Spearman")

plt.tight_layout()
plt.show()

## 4.4 Preprocesamiento

El preprocesamiento queda dentro de un objeto único que la validación cruzada reajusta en cada
partición, de modo que ninguna mediana ni ningún vocabulario aprendido con todos los datos
viaje a la partición de validación. El orden responde a dependencias entre pasos: las columnas
derivadas tienen que existir antes de que se rellenen sus vacíos, y el relleno por grupo
precede al escalado.

In [ ]:
fig, ax = ac.lienzo(12.5, 3.2)                                 # lienzo para el esquema

ac.caja(ax, 0.005, 0.35, 0.15, 0.40, "CrearColumnas", "banos, razones,\ndistancias, area", ac.ACENTO)
ac.caja(ax, 0.175, 0.35, 0.15, 0.40, "DummiesAmenities", "una columna por\ncomodidad frecuente", ac.ACENTO)
ac.caja(ax, 0.345, 0.35, 0.15, 0.40, "RellenarPorGrupo", "mediana por\ncapacidad", ac.ACENTO)
ac.caja(ax, 0.515, 0.56, 0.22, 0.30, "Numericas", "mediana y escalado", ac.MORADO)
ac.caja(ax, 0.515, 0.16, 0.22, 0.30, "Categoricas", "0/1, raras agrupadas", ac.MORADO)
ac.caja(ax, 0.775, 0.30, 0.21, 0.42, "Modelo", "sobre escala logaritmica", ac.TINTA)

ac.flecha(ax, (0.155, 0.55), (0.175, 0.55))
ac.flecha(ax, (0.325, 0.55), (0.345, 0.55))
ac.flecha(ax, (0.495, 0.55), (0.515, 0.70))
ac.flecha(ax, (0.495, 0.55), (0.515, 0.30))
ac.flecha(ax, (0.735, 0.70), (0.775, 0.56))
ac.flecha(ax, (0.735, 0.30), (0.775, 0.46))

ax.set_ylim(0.1, 0.95)
ax.set_title("Preprocesamiento, en el orden en que se aplica", loc="left", pad=10)
plt.show()

In [ ]:
preprocesador = ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)    # preprocesamiento con los bloques elegidos
t0 = time.time()
matriz_train = preprocesador.fit_transform(train, np.log1p(train["ingreso_anual"].fillna(0)))
print(f"Matriz de entrenamiento: {matriz_train.shape[0]:,} filas x {matriz_train.shape[1]} columnas")
print(f"Vacios que quedan:       {int(np.isnan(matriz_train).sum())}")
print(f"Comodidades abiertas:    {len(preprocesador.named_steps['dummies'].nombres_)}")
print(f"Tiempo:                  {time.time() - t0:.1f} s")

## 4.5 Aporte de cada bloque

La selección por mecanismo no garantiza que cada bloque aporte. La ablación añade los bloques de
uno en uno y mide el error con la misma validación cruzada agrupada, de modo que la decisión de
conservar un bloque descanse en la diferencia de error y no en el criterio con que se propuso.

La comparación se hace sobre el objetivo en escala logarítmica. En pesos, el error cuadrático
queda dominado por la cola derecha y las diferencias entre configuraciones se vuelven
indistinguibles de la variación entre particiones.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_validate  # validacion cruzada agrupada
from sklearn.ensemble import HistGradientBoostingRegressor       # potenciado por histogramas, rapido
from sklearn.pipeline import Pipeline                            # encadena preprocesamiento y modelo

cv = GroupKFold(n_splits=5)                                      # cinco particiones que respetan el anfitrion
medible_train = train[train["ingreso_anual"].notna()].copy()     # entrenamiento con el objetivo medido
grupos = medible_train["host_id"]                                # el anfitrion define el grupo
objetivo_log = np.log1p(medible_train["ingreso_anual"])          # objetivo en escala logaritmica

CONFIGURACIONES = [                                              # bloques acumulados
    ("1. Seleccion de referencia", ["referencia", "tipologia"]),
    ("2. Estructura y tipologia", ["estructura", "tipologia"]),
    ("3. + Ubicacion", ["estructura", "tipologia", "ubicacion"]),
    ("4. + Reglas de reserva", ["estructura", "tipologia", "ubicacion", "reglas"]),
    ("5. + Anfitrion", ["estructura", "tipologia", "ubicacion", "reglas", "decision_anfitrion"]),
    ("6. + Reglas del calendario", ["estructura", "tipologia", "ubicacion", "reglas",
                                     "decision_anfitrion", "reglas_calendario"]),
    ("7. + Comodidades", ["estructura", "tipologia", "ubicacion", "reglas", "decision_anfitrion",
                           "reglas_calendario", "equipamiento", "comodidades"]),
    ("8. + Presentacion y area", ac.BLOQUES_ELEGIDOS),
    ("9. + Texto del anuncio", ac.BLOQUES_ELEGIDOS + ["texto"]),
]

filas = []
for nombre, bloques in CONFIGURACIONES:                          # recorre las configuraciones
    t0 = time.time()
    pipe = Pipeline([("prep", ac.armar_preprocesador(bloques)),
                     ("modelo", HistGradientBoostingRegressor(random_state=ac.SEMILLA))])
    r = cross_validate(pipe, medible_train, objetivo_log, cv=cv, groups=grupos,
                       scoring=["r2", "neg_mean_absolute_error"], return_train_score=True)
    filas.append({"Configuracion": nombre,
                  "Columnas": ac.armar_preprocesador(bloques).fit_transform(medible_train, objetivo_log).shape[1],
                  "R2 validacion": r["test_r2"].mean(), "desv": r["test_r2"].std(),
                  "R2 entrenamiento": r["train_r2"].mean(),
                  "MAE log": -r["test_neg_mean_absolute_error"].mean(), "seg": time.time() - t0})
    print(f"  {nombre:30s} {filas[-1]['Columnas']:4d} columnas   R2 {filas[-1]['R2 validacion']:.4f}   {filas[-1]['seg']:4.0f} s", flush=True)

ablacion = pd.DataFrame(filas).set_index("Configuracion")
ablacion["Ganancia"] = ablacion["R2 validacion"].diff()          # cuanto suma cada bloque
ablacion["Brecha"] = ablacion["R2 entrenamiento"] - ablacion["R2 validacion"]   # distancia entre ajuste y validacion
print()
print(ablacion.round(4).to_string())
ac.guardar(ablacion, "ablacion_ingreso.csv")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4.2))            # dos paneles

posicion = np.arange(len(ablacion))
ejes[0].barh(posicion, ablacion["R2 validacion"], color=ac.ACENTO, height=0.6)
ejes[0].set_yticks(posicion); ejes[0].set_yticklabels(ablacion.index, fontsize=8.5)
ejes[0].invert_yaxis()
for i, valor in enumerate(ablacion["R2 validacion"]):           # etiqueta directa de cada barra
    ejes[0].text(valor + 0.004, i, f"{valor:.3f}", va="center", fontsize=8.5, color=ac.TINTA)
ejes[0].set_xlim(0, ablacion["R2 validacion"].max() * 1.18)
ejes[0].set_title("R2 en validacion cruzada agrupada")
ejes[0].set_xlabel("R2 sobre el objetivo en escala logaritmica")

ejes[1].plot(posicion, ablacion["R2 entrenamiento"], "o-", color=ac.MORADO, linewidth=2, markersize=6, label="entrenamiento")
ejes[1].plot(posicion, ablacion["R2 validacion"], "o-", color=ac.ACENTO, linewidth=2, markersize=6, label="validacion")
ejes[1].fill_between(posicion, ablacion["R2 validacion"], ablacion["R2 entrenamiento"], color=ac.GRIS, alpha=0.18)
ejes[1].set_xticks(posicion); ejes[1].set_xticklabels([n.split(".")[0] for n in ablacion.index], fontsize=9)
ejes[1].set_title("Distancia entre ajuste y validacion")
ejes[1].set_xlabel("configuracion"); ejes[1].set_ylabel("R2")
ejes[1].legend(loc="center right", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
mejor = ablacion["R2 validacion"].idxmax()                      # configuracion de mayor R2
referencia_r2 = ablacion.loc["1. Seleccion de referencia", "R2 validacion"]
print(f"Mejor configuracion: {mejor}")
print(f"  R2 {ablacion.loc[mejor, 'R2 validacion']:.4f} con {ablacion.loc[mejor, 'Columnas']} columnas,")
print(f"  frente a {referencia_r2:.4f} con {ablacion.loc['1. Seleccion de referencia', 'Columnas']} de la seleccion de referencia.")
print()
print("Aporte de cada bloque:")
print(ablacion["Ganancia"].dropna().sort_values(ascending=False).round(4).to_string())
print()
print("Bloques propuestos y descartados por la medicion:")
for nombre, motivo in ac.DESCARTES_EMPIRICOS.items():           # deja constancia de lo descartado
    print(f"  {nombre}: {motivo}")

### Cuántas comodidades conviene abrir

La lista de comodidades es una sola columna de texto que el preprocesamiento convierte en
muchas columnas de presencia. El umbral de frecuencia decide cuántas. Una comodidad que
aparece en un puñado de anuncios permite memorizar esas filas sin aportar nada general,
mientras que una presente en casi todos no distingue a un alojamiento de otro.

El criterio adoptado conserva las comodidades presentes en al menos uno de cada diez anuncios.
La curva de error frente al tamaño de la matriz muestra lo que cuesta cada recorte.

In [ ]:
UMBRALES = [0.03, 0.10, 0.20, 0.35]                             # fraccion minima de anuncios que debe tenerla
filas_parsimonia = []
for umbral in UMBRALES:                                          # recorre los umbrales candidatos
    pipe = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)),
                     ("modelo", HistGradientBoostingRegressor(random_state=ac.SEMILLA))])
    pipe.set_params(prep__dummies__frec_min=umbral)              # fija el umbral de esta prueba
    r = cross_validate(pipe, medible_train, objetivo_log, cv=cv, groups=grupos, scoring="r2")
    prep = ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)           # el mismo preprocesamiento, para contar columnas
    prep.set_params(dummies__frec_min=umbral)
    matriz = prep.fit_transform(medible_train, objetivo_log)
    filas_parsimonia.append({"Umbral": f"{umbral * 100:.0f} %",
                             "Comodidades": len(prep.named_steps["dummies"].nombres_),
                             "Columnas": matriz.shape[1],
                             "R2 validacion": r["test_score"].mean()})
    print(f"  presentes en mas del {umbral * 100:4.0f} %   "
          f"{filas_parsimonia[-1]['Comodidades']:3d} comodidades   "
          f"{filas_parsimonia[-1]['Columnas']:3d} columnas   R2 {filas_parsimonia[-1]['R2 validacion']:.4f}", flush=True)

sin_comodidades = [b for b in ac.BLOQUES_ELEGIDOS if b != "comodidades"]   # version sin abrirlas
r = cross_validate(Pipeline([("prep", ac.armar_preprocesador(sin_comodidades)),
                             ("modelo", HistGradientBoostingRegressor(random_state=ac.SEMILLA))]),
                   medible_train, objetivo_log, cv=cv, groups=grupos, scoring="r2")
matriz = ac.armar_preprocesador(sin_comodidades).fit_transform(medible_train, objetivo_log)
filas_parsimonia.append({"Umbral": "ninguna", "Comodidades": 0,
                         "Columnas": matriz.shape[1], "R2 validacion": r["test_score"].mean()})

parsimonia = pd.DataFrame(filas_parsimonia).set_index("Umbral")
print()
print(parsimonia.round(4).to_string())
ac.guardar(parsimonia, "parsimonia_comodidades.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))                        # una sola figura

ax.plot(parsimonia["Columnas"], parsimonia["R2 validacion"], "o-", color=ac.ACENTO,
        linewidth=2, markersize=8)
for umbral, fila in parsimonia.iterrows():                       # etiqueta cada punto con su umbral
    ax.annotate(f"{umbral}
{int(fila['Comodidades'])} comodidades",
                (fila["Columnas"], fila["R2 validacion"]), fontsize=8, color=ac.TINTA,
                xytext=(0, -28), textcoords="offset points", ha="center")

elegido = parsimonia.loc["10 %"]                                 # el umbral adoptado
ax.plot([elegido["Columnas"]], [elegido["R2 validacion"]], "o", color=ac.MORADO, markersize=15,
        markerfacecolor="none", markeredgewidth=2)
ax.annotate("umbral adoptado", (elegido["Columnas"], elegido["R2 validacion"]),
            fontsize=9, color=ac.MORADO, xytext=(12, 10), textcoords="offset points")

ax.set_title("Error y tamano de la matriz, segun el umbral de comodidades")
ax.set_xlabel("columnas de la matriz"); ax.set_ylabel("R2 en validacion cruzada agrupada")
ax.set_ylim(parsimonia["R2 validacion"].min() - 0.035, parsimonia["R2 validacion"].max() + 0.02)
plt.tight_layout()
plt.show()

In [ ]:
completo = parsimonia.loc["3 %"]                                 # la version mas ancha
adoptado = parsimonia.loc["10 %"]                                # la adoptada
ninguna = parsimonia.loc["ninguna"]                              # sin abrir las comodidades

print(f"Pasar de {int(completo['Comodidades'])} a {int(adoptado['Comodidades'])} comodidades reduce")
print(f"la matriz de {int(completo['Columnas'])} a {int(adoptado['Columnas'])} columnas y cuesta "
      f"{completo['R2 validacion'] - adoptado['R2 validacion']:.4f} de R2.")
print()
print(f"Prescindir del todo de las comodidades abiertas deja {int(ninguna['Columnas'])} columnas")
print(f"y cuesta {adoptado['R2 validacion'] - ninguna['R2 validacion']:.4f} de R2, que ya no es despreciable.")
print()
print("Las comodidades muy raras no cambian el error: solo ensanchan la matriz.")

---
# Paso 5 — Modelos

## 5.1 El diseño en tres etapas

Una regresión única sobre un objetivo con un tercio de ceros reparte la diferencia y asigna
valores intermedios a alojamientos que no registran ninguna venta. El diseño separa las dos
preguntas que el propietario tiene mezcladas, y las vuelve a unir al final.

La primera etapa estima con qué probabilidad el alojamiento llega a registrar actividad
detectable. La etiqueta honesta de esa etapa es la detección, no la venta, porque el método
reconoce la actividad a través de los comentarios. La segunda etapa estima cuánto rinde por día
de exposición un alojamiento que sí la registra. La tercera compone las dos y expresa el
resultado sobre un año.

In [ ]:
fig, ax = ac.lienzo(12, 3.2)                                   # lienzo para el esquema del diseno

ac.caja(ax, 0.01, 0.30, 0.17, 0.42, "Caracteristicas", "conocidas antes\nde publicar", ac.GRIS)
ac.caja(ax, 0.26, 0.56, 0.26, 0.34, "Etapa A — Riesgo", "probabilidad de\nactividad detectable", ac.MORADO)
ac.caja(ax, 0.26, 0.08, 0.26, 0.34, "Etapa B — Nivel", "ingreso por dia\nentre los activos", ac.ACENTO)
ac.caja(ax, 0.60, 0.30, 0.17, 0.42, "Etapa C", "producto\nx 365 dias", ac.TINTA)
ac.caja(ax, 0.82, 0.30, 0.17, 0.42, "Ingreso anual", "esperado", ac.TINTA)

ac.flecha(ax, (0.18, 0.58), (0.26, 0.72))
ac.flecha(ax, (0.18, 0.44), (0.26, 0.26))
ac.flecha(ax, (0.52, 0.72), (0.60, 0.58))
ac.flecha(ax, (0.52, 0.26), (0.60, 0.44))
ac.flecha(ax, (0.77, 0.51), (0.82, 0.51))

ax.set_title("Diseno en tres etapas", loc="left", pad=10)
plt.show()

## 5.2 Etapa A — Probabilidad de actividad detectable

In [ ]:
from sklearn.linear_model import LogisticRegression            # familia lineal para clasificacion
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier   # ensambles
from sklearn.metrics import roc_auc_score, roc_curve            # area bajo la curva y curva ROC

y_actividad = train["actividad_detectada"].astype(int)          # objetivo binario de la etapa A

CLASIFICADORES = {
    "Regresion logistica": LogisticRegression(max_iter=2000),
    "Random forest": RandomForestClassifier(n_estimators=300, random_state=ac.SEMILLA, n_jobs=-1),
    "Potenciado (histogramas)": HistGradientBoostingClassifier(random_state=ac.SEMILLA),
}
filas_a = []
for nombre, estimador in CLASIFICADORES.items():                # compara las tres familias
    t0 = time.time()
    pipe = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)), ("modelo", estimador)])
    r = cross_validate(pipe, train, y_actividad, cv=cv, groups=train["host_id"], scoring="roc_auc")
    filas_a.append({"Modelo": nombre, "AUC": r["test_score"].mean(),
                    "desv": r["test_score"].std(), "Tiempo (s)": time.time() - t0})
    print(f"  {nombre:28s} AUC {filas_a[-1]['AUC']:.4f}   {filas_a[-1]['Tiempo (s)']:4.0f} s", flush=True)

etapa_a = pd.DataFrame(filas_a).set_index("Modelo").sort_values("AUC", ascending=False)
print()
print(etapa_a.round(4).to_string())
print()
print(f"Proporcion de anuncios con actividad detectable: {y_actividad.mean() * 100:.1f} %")

In [ ]:
mejor_a = etapa_a.index[0]                                      # clasificador de mayor AUC
modelo_a = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)),
                     ("modelo", CLASIFICADORES[mejor_a])]).fit(train, y_actividad)
prob_test = modelo_a.predict_proba(test)[:, 1]                  # probabilidad en la particion de prueba
auc_test = roc_auc_score(test["actividad_detectada"].astype(int), prob_test)

fig, ejes = plt.subplots(1, 2, figsize=(11.5, 4))              # dos paneles

fpr, tpr, _ = roc_curve(test["actividad_detectada"].astype(int), prob_test)   # curva ROC en prueba
ejes[0].plot(fpr, tpr, color=ac.ACENTO, linewidth=2)
ejes[0].plot([0, 1], [0, 1], color=ac.GRIS, linewidth=1, linestyle="--")
ejes[0].set_title("Curva ROC en la particion de prueba")
ejes[0].set_xlabel("falsos positivos"); ejes[0].set_ylabel("verdaderos positivos")
ejes[0].text(0.45, 0.2, f"AUC {auc_test:.3f}", fontsize=11, color=ac.TINTA, fontweight="bold")

deciles = pd.qcut(prob_test, 10, labels=False, duplicates="drop")   # diez grupos de igual tamano
observado = pd.Series(test["actividad_detectada"].astype(int).to_numpy()).groupby(deciles).mean()
predicho = pd.Series(prob_test).groupby(deciles).mean()
ejes[1].plot(predicho.values, observado.values, "o-", color=ac.MORADO, linewidth=2, markersize=7, label="observado")
ejes[1].plot([0, 1], [0, 1], color=ac.GRIS, linewidth=1, linestyle="--", label="calibracion perfecta")
ejes[1].set_title("Calibracion por decil de probabilidad")
ejes[1].set_xlabel("probabilidad predicha"); ejes[1].set_ylabel("proporcion observada")
ejes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5.3 Etapa B — Ingreso por día de exposición

La segunda etapa se ajusta sobre los alojamientos con exposición observable y actividad
detectada, y predice el ingreso por día expuesto. Expresar el objetivo como tasa restituye la
comparabilidad entre alojamientos publicados en momentos distintos.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso   # familia lineal
from sklearn.neighbors import KNeighborsRegressor                 # vecinos mas cercanos
from sklearn.tree import DecisionTreeRegressor                    # arbol de decision
from sklearn.neural_network import MLPRegressor                   # red neuronal
from sklearn.compose import TransformedTargetRegressor            # ajusta sobre el objetivo transformado
from xgboost import XGBRegressor                                  # potenciado por gradiente de XGBoost

activos = train[train["exposicion_observable"] & train["actividad_detectada"]].copy()   # muestra de la etapa B
print(f"Muestra de la etapa B: {len(activos):,} anuncios de {activos['host_id'].nunique():,} anfitriones")
print(f"Ingreso por dia expuesto, mediana: ${activos['ingreso_por_dia'].median():,.0f}")

def modelo_tasa(estimador):                                       # arma el Pipeline de un modelo de la etapa B
    base = Pipeline([("prep", ac.armar_preprocesador(ac.BLOQUES_ELEGIDOS)), ("modelo", estimador)])
    return TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)

MODELOS = {
    "Regresion lineal": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001, max_iter=5000),
    "KNN (15 vecinos)": KNeighborsRegressor(n_neighbors=15),
    "Arbol (profundidad 8)": DecisionTreeRegressor(max_depth=8, random_state=ac.SEMILLA),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=ac.SEMILLA, n_jobs=-1),
    "Potenciado (histogramas)": HistGradientBoostingRegressor(random_state=ac.SEMILLA),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.9,
                            colsample_bytree=0.9, random_state=ac.SEMILLA, n_jobs=-1, verbosity=0),
    "Red neuronal (64-32)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=400,
                                         random_state=ac.SEMILLA, early_stopping=True),
}
print(f"Familias comparadas: {len(MODELOS)}, mas la linea base y la regla a mano")

In [ ]:
METRICAS = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}

def evaluar(nombre, estimador, tabla, objetivo):                 # mide un modelo con la validacion agrupada
    t0 = time.time()
    r = cross_validate(estimador, tabla, objetivo, cv=cv, groups=tabla["host_id"], scoring=METRICAS)
    print(f"  {nombre:28s} listo en {time.time() - t0:5.1f} s", flush=True)
    return {"Modelo": nombre, "MAE": -r["test_mae"].mean(), "RMSE": -r["test_rmse"].mean(),
            "R2": r["test_r2"].mean(), "desv R2": r["test_r2"].std(), "Tiempo (s)": time.time() - t0}

y_tasa = activos["ingreso_por_dia"]                              # objetivo de la etapa B, en pesos por dia
resultados = [
    evaluar("Linea base (mediana)", DummyRegressor(strategy="median"), activos, y_tasa),
    evaluar("Regla a mano (localidad)", ac.ReglaPorGrupo("neighbourhood_cleansed"), activos, y_tasa),
]
for nombre, estimador in MODELOS.items():
    resultados.append(evaluar(nombre, modelo_tasa(estimador), activos, y_tasa))

etapa_b = pd.DataFrame(resultados).set_index("Modelo").sort_values("MAE")
print()
print(etapa_b.round(3).to_string())
ac.guardar(etapa_b, "etapa_b_modelos.csv")

### Lo que cuesta responderle a un propietario nuevo

El modelo que informa a quien todavía no publicó renuncia a las variables de desempeño. Medir
esa renuncia pone precio a la honestidad del diseño, y evita que la cifra del escenario de
operador se presente como si valiera para los dos casos.

In [ ]:
escenarios = {}
for etiqueta, bloques in [("Propietario nuevo", ac.BLOQUES_ELEGIDOS),
                          ("Operador con historial", ac.BLOQUES_OPERADOR)]:
    r = cross_validate(Pipeline([("prep", ac.armar_preprocesador(bloques)),
                                 ("modelo", HistGradientBoostingRegressor(random_state=ac.SEMILLA))]),
                       activos, np.log1p(activos["ingreso_por_dia"]), cv=cv,
                       groups=activos["host_id"], scoring="r2")
    escenarios[etiqueta] = r["test_score"].mean()
    print(f"  {etiqueta:26s} R2 en escala logaritmica {r['test_score'].mean():.4f} (±{r['test_score'].std():.4f})", flush=True)

costo = escenarios["Operador con historial"] - escenarios["Propietario nuevo"]
print()
print(f"Excluir las variables de desempeno cuesta {costo:.4f} de R2.")
print("Ese es el precio de responder la pregunta que el propietario nuevo hace de verdad,")
print("en lugar de la que resulta mas facil de ajustar.")

In [ ]:
orden_b = etapa_b.sort_values("R2")                              # de menor a mayor R2
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6))

colores = [ac.ACENTO if n == etapa_b["R2"].idxmax() else ac.GRIS for n in orden_b.index]
ejes[0].barh(orden_b.index, orden_b["R2"], color=colores, height=0.62)
ejes[0].errorbar(orden_b["R2"], np.arange(len(orden_b)), xerr=orden_b["desv R2"],
                 fmt="none", ecolor=ac.TINTA, elinewidth=1, capsize=2.5)
ejes[0].axvline(0, color=ac.TINTA, linewidth=0.8)
ejes[0].set_title("R2 en validacion cruzada agrupada")
ejes[0].set_xlabel("R2 del ingreso por dia expuesto")
ejes[0].tick_params(axis="y", labelsize=8.5)

ejes[1].scatter(etapa_b["Tiempo (s)"], etapa_b["MAE"], s=42, color=ac.MORADO, zorder=3)
for nombre, fila in etapa_b.iterrows():                          # etiqueta cada punto
    ejes[1].annotate(nombre, (fila["Tiempo (s)"], fila["MAE"]), fontsize=7.4,
                     xytext=(5, 3), textcoords="offset points", color=ac.TINTA)
ejes[1].set_xscale("log")
ejes[1].set_title("Error y costo de ajuste")
ejes[1].set_xlabel("segundos de validacion cruzada, escala log")
ejes[1].set_ylabel("MAE en pesos por dia")

plt.tight_layout()
plt.show()

## 5.4 Ajuste de los parámetros del finalista

La búsqueda aleatoria recorre mejor el espacio que una rejilla completa cuando unos pocos
parámetros concentran el efecto, porque no gasta ajustes repitiendo el valor del parámetro que
importa.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV          # busqueda aleatoria de parametros

finalista = etapa_b.drop(["Linea base (mediana)", "Regla a mano (localidad)"]).sort_values("MAE").index[0]
print("Modelo con menor error en la etapa B:", finalista)

REJILLAS = {
    "Random forest": ({"regressor__modelo__n_estimators": [300, 500],
                       "regressor__modelo__max_features": [0.2, 0.4, 0.7, 1.0],
                       "regressor__modelo__min_samples_leaf": [1, 2, 5, 10]},
                      RandomForestRegressor(random_state=ac.SEMILLA, n_jobs=-1)),
    "XGBoost": ({"regressor__modelo__n_estimators": [300, 500, 800],
                 "regressor__modelo__learning_rate": [0.03, 0.05, 0.08],
                 "regressor__modelo__max_depth": [4, 6, 8],
                 "regressor__modelo__subsample": [0.7, 0.9, 1.0]},
                XGBRegressor(random_state=ac.SEMILLA, n_jobs=-1, verbosity=0)),
}
rejilla, estimador_base = REJILLAS.get(finalista, (
    {"regressor__modelo__max_iter": [200, 300, 400, 600],
     "regressor__modelo__learning_rate": [0.03, 0.05, 0.08, 0.12],
     "regressor__modelo__max_leaf_nodes": [15, 31, 63],
     "regressor__modelo__min_samples_leaf": [10, 20, 40]},
    HistGradientBoostingRegressor(random_state=ac.SEMILLA)))

t0 = time.time()
busqueda = RandomizedSearchCV(modelo_tasa(estimador_base), rejilla, n_iter=12,
                              cv=GroupKFold(n_splits=3), scoring="neg_mean_absolute_error",
                              random_state=ac.SEMILLA, n_jobs=1)
busqueda.fit(activos, y_tasa, groups=activos["host_id"])
print(f"Combinaciones evaluadas: {len(busqueda.cv_results_['params'])} en {time.time() - t0:.0f} s")
print(f"MAE con parametros por defecto: ${etapa_b.loc[finalista, 'MAE']:,.0f} por dia")
print(f"MAE con parametros ajustados:   ${-busqueda.best_score_:,.0f} por dia")
print()
for clave, valor in busqueda.best_params_.items():
    print(f"  {clave.split('__')[-1]:20s} {valor}")

---
# Paso 6 — Evaluación

## 6.1 Composición de las dos etapas y medida final

La partición de prueba se usa una sola vez, con las dos etapas ya elegidas y ajustadas. La
predicción del ingreso anual es el producto de la probabilidad de actividad detectable por el
ingreso diario esperado, extendido a trescientos sesenta y cinco días.

In [ ]:
modelo_b = busqueda.best_estimator_                             # etapa B con sus parametros ajustados
tasa_test = np.clip(modelo_b.predict(test), 0, None)            # ingreso diario esperado en prueba
prediccion = prob_test * tasa_test * 365                        # composicion de las dos etapas

evaluable = test["ingreso_anual"].notna()                       # filas de prueba con el objetivo medido
y_test = test.loc[evaluable, "ingreso_anual"]                   # objetivo observado

final = pd.DataFrame({
    "Linea base": ac.metricas(y_test, base.predict(test[evaluable])),
    "Regla a mano": ac.metricas(y_test, regla.predict(test[evaluable])),
    "Diseno en tres etapas": ac.metricas(y_test, prediccion[evaluable.to_numpy()]),
}).T
print(final.to_string())

mejora = (1 - final.loc["Diseno en tres etapas", "MAE"] / final.loc["Regla a mano", "MAE"]) * 100
print()
print(f"Sobre la particion de prueba, el diseno reduce el error absoluto medio un {mejora:.1f} %")
print(f"frente a la regla a mano, de ${final.loc['Regla a mano', 'MAE']:,.0f} a ${final.loc['Diseno en tres etapas', 'MAE']:,.0f} al ano.")
ac.guardar(final, "evaluacion_directo.csv")

## 6.2 Importancia por permutación

La importancia por permutación mide cuánto empeora el error al desordenar una sola columna, de
modo que recoge el aporte de la variable al modelo ya ajustado y no su asociación bruta con el
objetivo. Comparar el resultado con el signo declarado en el paso cuatro cierra el
procedimiento.

In [ ]:
from sklearn.inspection import permutation_importance          # importancia por permutacion

t0 = time.time()
muestra = activos.sample(min(1200, len(activos)), random_state=ac.SEMILLA)   # submuestra, el calculo repite el ajuste
importancia = permutation_importance(modelo_b, muestra, muestra["ingreso_por_dia"],
                                     n_repeats=5, random_state=ac.SEMILLA,
                                     scoring="neg_mean_absolute_error")
pesos = pd.Series(importancia.importances_mean, index=muestra.columns)
pesos = pesos[pesos > 0].sort_values(ascending=False)
print(f"Calculo: {time.time() - t0:.0f} s")
print()
print("Columnas de entrada mas importantes:")
print(pesos.head(15).round(1).to_string())

In [ ]:
top = pesos.head(15).iloc[::-1]                                 # las 15 mas importantes
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.hlines(top.index, 0, top.values, color=ac.GRIS, linewidth=1.2)
ax.plot(top.values, top.index, "o", color=ac.ACENTO, markersize=8)
ax.set_title("Importancia por permutacion en la etapa de nivel")
ax.set_xlabel("aumento del error absoluto medio al desordenar la columna, pesos por dia")
plt.tight_layout()
plt.show()

## 6.3 Dónde se equivoca el modelo

In [ ]:
errores = pd.DataFrame({
    "real": y_test.to_numpy(),
    "predicho": prediccion[evaluable.to_numpy()],
    "room_type": test.loc[evaluable, "room_type"].to_numpy(),
    "noches": test.loc[evaluable, "noches_ocupadas"].to_numpy(),
}, index=y_test.index)
errores["error_abs"] = (errores["real"] - errores["predicho"]).abs()
errores["estado"] = np.where(errores["noches"] == 0, "Sin actividad",
                      np.where(errores["noches"] >= 255, "En el tope", "Con actividad"))

print("Error por tipo de alojamiento:")
print(errores.groupby("room_type").agg(alojamientos=("real", "size"), ingreso_mediano=("real", "median"),
                                       MAE=("error_abs", "mean")).sort_values("alojamientos", ascending=False).round(0).to_string())
print()
print("Error por estado del anuncio:")
por_estado = errores.groupby("estado").agg(alojamientos=("real", "size"), ingreso_mediano=("real", "median"),
                                           prediccion_mediana=("predicho", "median"), MAE=("error_abs", "mean"))
print(por_estado.round(0).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))

limite = float(np.percentile(errores["real"], 99))
ejes[0].scatter(errores["real"], errores["predicho"], s=7, alpha=0.25, color=ac.MORADO, edgecolors="none")
ejes[0].plot([0, limite], [0, limite], color=ac.TINTA, linewidth=1, linestyle="--")
ejes[0].set_xlim(0, limite); ejes[0].set_ylim(0, limite)
ejes[0].set_title("Ingreso predicho y observado")
ejes[0].set_xlabel("ingreso anual observado"); ejes[0].set_ylabel("ingreso anual predicho")
ejes[0].xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[0].yaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[0].text(limite * 0.04, limite * 0.92, "prediccion perfecta", fontsize=8, color=ac.TINTA)

decil = pd.qcut(errores["predicho"], 10, labels=False, duplicates="drop")
calibracion = errores.groupby(decil).agg(observado=("real", "median"), predicho=("predicho", "median"))
ejes[1].plot(calibracion["predicho"], calibracion["observado"], "o-", color=ac.ACENTO, linewidth=2, markersize=7, label="observado")
tope_cal = calibracion[["observado", "predicho"]].max().max()
ejes[1].plot([0, tope_cal], [0, tope_cal], color=ac.GRIS, linewidth=1, linestyle="--", label="calibracion perfecta")
ejes[1].set_title("Calibracion por decil de prediccion")
ejes[1].set_xlabel("ingreso predicho, mediana del decil"); ejes[1].set_ylabel("ingreso observado, mediana del decil")
ejes[1].xaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[1].yaxis.set_major_formatter(plt.FuncFormatter(ac.miles))
ejes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
ejemplos = test[evaluable].sample(6, random_state=ac.SEMILLA)   # seis anuncios de prueba al azar
indices = [test.index.get_loc(i) for i in ejemplos.index]        # posicion de cada uno en la prediccion
tabla = pd.DataFrame({
    "tipo": ejemplos["room_type"],
    "huespedes": ejemplos["accommodates"],
    "localidad": ejemplos["neighbourhood_cleansed"],
    "tarifa": ejemplos["precio_noche"].round(0),
    "noches": ejemplos["noches_ocupadas"],
    "prob. actividad": np.round(prob_test[indices], 2),
    "ingreso real": ejemplos["ingreso_anual"].round(0),
    "ingreso predicho": np.round(prediccion[indices], 0),
})
print(tabla.to_string())

## 6.4 Respuesta a la pregunta de investigación

In [ ]:
print("PREGUNTA 3. Dadas las caracteristicas de una vivienda, cual puede ser su rentabilidad anual.")
print("=" * 100)
print()
print(f"Sobre la particion de prueba, con anfitriones que el modelo no vio, el diseno en tres")
print(f"etapas predice el ingreso bruto anual con un error absoluto medio de ${final.loc['Diseno en tres etapas', 'MAE']:,.0f}")
print(f"y un R2 de {final.loc['Diseno en tres etapas', 'R2']:.3f}. El ingreso mediano observado es de ${y_test.median():,.0f}.")
print()
print(f"La regla a mano de la mediana por localidad se equivoca en ${final.loc['Regla a mano', 'MAE']:,.0f},")
print(f"de modo que el diseno reduce el error un {mejora:.1f} %.")
print()
print(f"La etapa de riesgo separa los anuncios que llegan a registrar actividad con un AUC de {auc_test:.3f}.")
print()
print("Error segun el estado del anuncio:")
for estado, fila in por_estado.iterrows():
    print(f"  {estado:16s} {int(fila['alojamientos']):5d} anuncios   "
          f"observado ${fila['ingreso_mediano']:>13,.0f}   MAE ${fila['MAE']:>13,.0f}")
print()
print("Variables de mayor aporte:")
for nombre, valor in pesos.head(6).items():
    print(f"  {nombre:42s} {valor:>10,.0f} pesos por dia de deterioro")

## 6.5 El techo de lo predecible

Queda una pregunta que el error por sí solo no responde: cuánto margen de mejora existe. Los
anuncios gemelos permiten estimarlo sin suponer nada sobre el modelo.

Dos anuncios del mismo anfitrión con igual tipo de alojamiento, capacidad, número de
habitaciones y localidad comparten todo lo que el modelo observa. Cualquier diferencia de
ingreso entre ellos es, para este conjunto de variables, inexplicable: procede del piso, de la
vista, de las fotografías, de cómo gestiona el anfitrión cada anuncio, y del ruido del método
que estima la ocupación a partir de unos pocos comentarios. La proporción de la varianza que
sobrevive dentro de los grupos de gemelos marca el techo de cualquier modelo construido con
estas variables.

In [ ]:
LLAVE = ["host_id", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed"]   # define un gemelo
train["grupo_gemelos"] = train.groupby(LLAVE, dropna=False).ngroup()                      # identifica cada grupo
tamanos_grupo = train["grupo_gemelos"].map(train["grupo_gemelos"].value_counts())         # tamano del grupo de cada fila
gemelos = train[tamanos_grupo >= 2]                                                        # solo los que tienen gemelo

print(f"Anuncios con al menos un gemelo: {len(gemelos):,} de {len(train):,} ({len(gemelos) / len(train) * 100:.1f} %)")
print(f"Grupos de gemelos: {gemelos['grupo_gemelos'].nunique():,}")
print()

def techo_r2(tabla, columna, etiqueta):                        # estima el techo de R2 de una variable
    t = tabla[tabla[columna].notna()]                          # filas con el valor medido
    t = t[t.groupby("grupo_gemelos")[columna].transform("size") >= 2]   # grupos que conservan al menos dos
    y = np.log1p(t[columna])                                    # objetivo en escala logaritmica
    total = y.var(ddof=1)                                       # varianza total
    centrada = y.groupby(t["grupo_gemelos"]).transform(lambda s: s - s.mean())   # desviacion dentro de cada grupo
    dentro = centrada.pow(2).sum() / (len(y) - t["grupo_gemelos"].nunique())      # varianza dentro de los grupos
    techo = 1 - dentro / total                                  # proporcion explicable como maximo
    print(f"  {etiqueta:36s} n={len(y):6,d}   techo de R2 ~ {techo:.3f}")
    return techo

print("Techo estimado, con el objetivo en escala logaritmica:")
techo_tarifa = techo_r2(gemelos, "precio_noche", "tarifa por noche")
techo_ingreso = techo_r2(gemelos, "ingreso_anual", "ingreso anual")
activos_gemelos = gemelos[gemelos["exposicion_observable"] & gemelos["actividad_detectada"]]
techo_tasa = techo_r2(activos_gemelos, "ingreso_por_dia", "ingreso por dia, solo activos")

dispersion = activos_gemelos.groupby("grupo_gemelos")["noches_ocupadas"].agg(["mean", "std", "size"])
dispersion = dispersion[dispersion["size"] >= 2]
print()
print(f"Entre gemelos activos, la desviacion mediana de las noches ocupadas es de "
      f"{dispersion['std'].median():.0f} noches")
print(f"sobre una media mediana de {dispersion['mean'].median():.0f}, un coeficiente de variacion del "
      f"{(dispersion['std'] / dispersion['mean']).median() * 100:.0f} %.")

In [ ]:
alcanzado = escenarios["Propietario nuevo"]                     # R2 logra el modelo del propietario nuevo
fig, ax = plt.subplots(figsize=(8.5, 2.6))                      # una sola figura, baja

ax.barh([0], [techo_tasa], color=ac.GRIS, height=0.45, label="techo con estas variables")
ax.barh([0], [alcanzado], color=ac.ACENTO, height=0.45, label="alcanzado por el modelo")
ax.set_yticks([]); ax.set_xlim(0, 1)
ax.set_title("Ingreso por dia expuesto: lo alcanzado y el techo")
ax.set_xlabel("R2 sobre el objetivo en escala logaritmica")
ax.text(alcanzado / 2, 0, f"{alcanzado:.3f}", ha="center", va="center", color="white", fontweight="bold")
ax.text((alcanzado + techo_tasa) / 2, 0, f"margen {techo_tasa - alcanzado:.3f}", ha="center", va="center",
        fontsize=9, color=ac.TINTA)
ax.text(techo_tasa + 0.02, 0, f"techo {techo_tasa:.3f}", va="center", fontsize=9, color=ac.TINTA)
ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout()
plt.show()

El techo acota la ambición del proyecto con una cifra en lugar de una impresión. Una parte
grande de la variación del ingreso no está en ninguna columna de la fuente, y buena parte de
esa parte ni siquiera está en el fenómeno: procede de estimar la ocupación a partir del
recuento de comentarios de cada alojamiento, con una tasa de reseña supuesta.

El techo corresponde a este conjunto de variables y no a un límite del problema. Información
genuinamente nueva, como las fotografías del anuncio, la dirección exacta o el tiempo de
respuesta del anfitrión, lo desplazaría hacia arriba.

# Conclusiones

El ingreso anual de un alojamiento resulta predecible a partir de características conocidas
antes de operar, con una precisión que depende del segmento y que es menor de lo que sugiere
una validación aleatoria. La predicción distingue con holgura un apartamento completo céntrico
de una habitación compartida periférica, y pierde capacidad en los dos extremos de la
distribución, donde conviven alojamientos sin ninguna venta detectada y alojamientos con el
calendario lleno.

La dificultad no se reparte por igual entre los dos factores del ingreso. La tarifa responde a
atributos observables del inmueble, mientras que la ocupación depende de decisiones de gestión
y de la posición en el buscador, que las características estáticas recogen de forma indirecta a
través de la profesionalización del anfitrión y de las reglas de reserva.

La separación entre riesgo y nivel recupera buena parte de la capacidad predictiva que una
regresión única pierde al promediar poblaciones con comportamiento distinto, y además entrega
dos cifras que el propietario necesita por separado: con qué probabilidad su alojamiento
consigue venderse, y cuánto rinde si lo consigue.

**Limitaciones.** El objetivo es una estimación de la fuente y no un ingreso contable; hereda el
supuesto de tasa de reseña con que se construye, y su techo censura a los alojamientos más
activos. El cero no distingue al anuncio nuevo del abandonado, porque el archivo no trae la
fecha de publicación. El segmento de tarifa mensual queda fuera del modelo de ingreso por estar
mal medido en los dos ejes. La medida es de ingreso bruto, de modo que la comparación con el
arriendo tradicional exigiría descontar la operación, el aseo, los servicios y la vacancia,
además de un precio de referencia del arriendo residencial que la fuente no contiene. Los datos
corresponden a una sola extracción, por lo que el análisis describe un corte transversal y no
permite separar la temporada de la anticipación de la reserva.